<a href="https://colab.research.google.com/github/weaamasad99/CloudComputingWolf/blob/main/HW2_Wolf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title  Index
# Install required libraries and import modules
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict
import nltk
from nltk.stem import PorterStemmer

# Download basic NLTK data required for stemming (if not already cached in Colab)
nltk.download('punkt')


# @title  Configuration & Global Variables
#adding the DB
FIREBASE_URL = "https://lemonpulse-index-default-rtdb.europe-west1.firebasedatabase.app/lemon_disease_index.json"

# The 5 academic articles related to Citrus / Lemon diseases
ARTICLE_URLS = [
    "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0316081",
    "https://link.springer.com/article/10.1007/s13593-014-0246-1",
    "https://link.springer.com/article/10.1007/BF02215619",
    "https://ieeexplore.ieee.org/document/9481921",
    "https://www.techscience.com/cmc/v66n1/40458"
]


# @title  Text Processing Functions

def fetch_and_extract_content(url):
    """Fetch an article page and extract abstract/introduction text."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 11.0; Win64; x64) AppleWebKit/537.36"
    }
    content = ""
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")

            # Look for common abstract or introduction classes/tags
            for selector in ['div.abstract', 'section.Abstract', 'div.introduction', 'p']:
                elements = soup.select(selector)
                for element in elements[:15]: # Limit to first few paragraphs to get core content
                    text = element.get_text(strip=True)
                    if text:
                        content += " " + text
        return content
    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return None

def get_stop_words():
      """Return an extended stop-word list to filter out noise."""
      basic_english = {
          "the", "is", "at", "which", "on", "and", "of", "to", "in", "a", "for", "with",
          "as", "by", "an", "that", "from", "this", "it", "are", "was", "were", "have",
          "has", "had", "but", "not", "can", "may", "will", "would", "could", "your", "they",
          "them", "their", "these", "those", "such", "very", "much", "many", "some",
          "we", "our", "all", "its", "also", "or", "if", "than", "then", "like", "into", "about",
          "how", "what", "when", "where", "who", "why", "any", "no", "only", "other", "so","plo", "sci", "too"
      }

      # (Academic & Research) stop words
      academic_terms = {
          "study", "paper", "research", "method", "methods", "result", "results",
          "analysis", "show", "use", "used", "using", "based", "data", "system", "approaches",
          "article", "articl", "response", "respons", "conclusion", "introduction", "abstract",
          "figure", "table", "section", "journal", "review", "literature", "proposed",
          "performance", "experiment", "model", "equation", "fig", "et", "al"
      }

      # General Noise & Time) stop words
      general_noise = {
          "new", "different", "important", "more", "etc", "year", "years", "high", "low",
          "large", "small", "good", "better", "modern", "well", "however", "thus", "therefore",
          "furthermore", "time", "day", "month", "first", "second", "one", "two", "three",
          "number", "value", "significant", "increase", "decrease", "within", "between"
      }

      return basic_english | academic_terms | general_noise

def clean_text(text):
    """Tokenize text, remove stop-words, apply Porter Stemmer, and return cleaned terms."""
    if not text:
        return []

    # Extract alphabetic words only, length >= 3
    words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
    stop_words = get_stop_words()
    stemmer = PorterStemmer()

    cleaned_words = []
    for word in words:
        if word not in stop_words:
            stemmed = stemmer.stem(word)
            if stemmed not in stop_words and len(stemmed) >= 3:
                cleaned_words.append(stemmed)

    return cleaned_words



    # @title TextRank & Indexing Logic
# TextRank Algorithm and Inverted Index Builders

def build_word_graph(words, window_size=4):
    """Build a co-occurrence graph for the TextRank algorithm."""
    graph = defaultdict(lambda: defaultdict(float))

    # Iterate through all words to establish connections based on proximity
    for i, word in enumerate(words):
        start = max(0, i - window_size)
        end = min(len(words), i + window_size + 1)

        for j in range(start, end):
            if i != j:
                neighbor = words[j]
                distance = abs(i - j)
                # Words that are closer to each other get a stronger weight (1/distance)
                graph[word][neighbor] += 1.0 / distance
    return graph

def calculate_textrank_scores(graph, damping=0.85, iterations=20):
    """Compute importance scores for each word using TextRank."""
    words = list(graph.keys())
    if not words:
        return {}

    # Initialize all nodes (words) with a base score of 1.0
    scores = {word: 1.0 for word in words}

    # Run power-iteration to converge the scores (similar to PageRank)
    for _ in range(iterations):
        new_scores = {}
        for word in words:
            # Base probability of jumping to a random node
            score = 1.0 - damping

            # Add distributed score from all connected neighbors
            for neighbor, connection_strength in graph[word].items():
                neighbor_total_connections = sum(graph[neighbor].values())
                if neighbor_total_connections > 0:
                    contribution = (connection_strength / neighbor_total_connections) * scores[neighbor]
                    score += damping * contribution
            new_scores[word] = score

        # Update scores for the next iteration
        scores = new_scores
    return scores

def count_word_frequencies(documents):
    """Create the inverted index mapping: term -> {doc_id: frequency_count}."""
    word_doc_counts = defaultdict(lambda: defaultdict(int))

    # Process each document and count term occurrences
    for doc_id, document in enumerate(documents, 1):
        words = clean_text(document)
        for word in words:
            word_doc_counts[word][doc_id] += 1

    return word_doc_counts

def extract_top_keywords(documents, top_k=30):
    """Combine documents, run TextRank, and return the top K keywords."""
    combined_text = " ".join(documents)
    words = clean_text(combined_text)

    # Fallback if the extracted content is extremely short
    if len(words) < 10:
        return words[:top_k]

    # Build the graph and calculate word importance
    graph = build_word_graph(words)
    scores = calculate_textrank_scores(graph)

    # Sort words by their TextRank score in descending order
    ranked_words = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # Return only the string values of the top 'K' words
    return [word for word, score in ranked_words[:top_k]]


    # @title Execution & Firebase Upload
# Main Execution Pipeline

def create_firebase_payload(top_keywords, word_doc_counts):
    """Format the inverted index into a JSON-friendly structure for Firebase."""
    firebase_data = {}

    for keyword in top_keywords:
        # Initialize the schema for each specific keyword
        firebase_data[keyword] = {
            "term": keyword,
            "DocsIds": {}
        }

        # Map document IDs to their term frequency.
        # Note: Firebase requires dictionary keys to be strings.
        for doc_id, count in word_doc_counts[keyword].items():
            firebase_data[keyword]["DocsIds"][str(doc_id)] = count

    return firebase_data

def upload_to_firebase(data, url):
    """Upload the final index to Firebase via PUT request."""
    try:
        # Using PUT will overwrite any existing data at this specific database node
        response = requests.put(url, json=data)
        if response.status_code in [200, 201]:
            print("\n[SUCCESS] Inverted Index uploaded to Firebase successfully!")
        else:
            print(f"\n[ERROR] Upload failed. Status Code: {response.status_code}")
    except Exception as e:
        print(f"\n[ERROR] Firebase connection error: {e}")


# Run Pipeline
documents = []
print("Fetching and processing articles...")

# Step 1: Fetch and clean content from all URLs
for i, url in enumerate(ARTICLE_URLS, 1):
    content = fetch_and_extract_content(url)
    # Ensure we actually scraped meaningful text (more than 50 chars)
    if content and len(content) > 50:
        documents.append(content)
        print(f" - Document {i} processed successfully.")
    else:
        print(f" - Document {i} could not be parsed (might be blocked/paywalled), skipping.")

# Step 2: If we have valid documents, build the index
if documents:
    print(f"\nExtracting keywords using TextRank from {len(documents)} documents...")

    # Extract keywords and calculate frequencies
    keywords = extract_top_keywords(documents, top_k=30)
    word_doc_counts = count_word_frequencies(documents)

    # Format the data for NoSQL DB storage
    firebase_data = create_firebase_payload(keywords, word_doc_counts)

    # Display the final results to the console
    print("\n--- TOP TEXTRANK KEYWORDS FOUND ---")
    for i, keyword in enumerate(keywords, 1):
        doc_distribution = ", ".join([f"Doc {doc_id}:{count}" for doc_id, count in firebase_data[keyword]["DocsIds"].items()])
        print(f"{i:2d}. {keyword} --> {doc_distribution}")

    # Step 3: Push the data to the cloud
    upload_to_firebase(firebase_data, FIREBASE_URL)
else:
    print("\n[FAILED] No content was extracted from the URLs.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Fetching and processing articles...
 - Document 1 processed successfully.
 - Document 2 processed successfully.
 - Document 3 processed successfully.
 - Document 4 processed successfully.
 - Document 5 processed successfully.

Extracting keywords using TextRank from 5 documents...

--- TOP TEXTRANK KEYWORDS FOUND ---
 1. diseas --> Doc 1:4, Doc 2:6, Doc 5:2
 2. fruit --> Doc 1:2, Doc 3:8
 3. irrig --> Doc 3:8
 4. citru --> Doc 1:3, Doc 3:4
 5. treatment --> Doc 3:8
 6. plant --> Doc 2:5, Doc 3:2
 7. effect --> Doc 1:1, Doc 2:3, Doc 3:2
 8. water --> Doc 3:6
 9. detect --> Doc 1:1, Doc 2:4, Doc 5:1
10. yield --> Doc 1:1, Doc 3:5
11. technolog --> Doc 1:2, Doc 2:1, Doc 4:1, Doc 5:2
12. univers --> Doc 1:3, Doc 5:3
13. access --> Doc 1:1, Doc 2:1, Doc 3:2, Doc 5:1
14. lemon --> Doc 1:1, Doc 3:4
15. learn --> Doc 1:3, Doc 5:1
16. comput --> Doc 1:3, Doc 5:2
17. period --> Doc 3:5
18. growth --> Doc 3:5
19. dna --> Doc 2:4
20. leaf --> Doc 3:2, Doc 5:2
21. agricultur --> Doc 1:2, Doc 2:2
22

In [ ]:

# @title Application

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
import io
import base64
import random
import time
import requests
import re
from nltk.stem import PorterStemmer
from collections import defaultdict

# 1.  APPLICATION UI STYLING (CSS)
css_style = """
<style>
    @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700;800&display=swap');

    .lemon-app {
        font-family: 'Outfit', sans-serif !important;
        background-color: #F8FAFC !important;
        max-width: 1400;
        margin: 20px auto;
        border-radius: 12px !important;
        box-shadow: 0 4px 12px rgba(0,0,0,0.06), 0 1px 3px rgba(0,0,0,0.02) !important;
        overflow: hidden !important;
        border: 1px solid #E2E8F0 !important;
    }

    /* Force light background on Jupyter widgets to hide Dark Mode bleed-through */
    .lemon-app,
    .lemon-app .widget-box,
    .lemon-app .widget-vbox,
    .lemon-app .widget-output {
        background-color: #F8FAFC !important;
    }

    .app-container {
        padding: 24px;
        min-height: 520px;
        background: #F8FAFC;
    }
    .app-header {
        margin-bottom: 24px;
    }
    .app-title {
        font-size: 26px;
        font-weight: 800;
        color: #0F172A;
        margin: 0;
        letter-spacing: -0.5px;
    }
    .app-subtitle {
        font-size: 13px;
        color: #64748B;
        margin-top: 6px;
        font-weight: 500;
    }
    .main-card {
        background: #FFFFFF;
        border-radius: 22px;
        padding: 20px;
        box-shadow: 0 4px 20px rgba(0,0,0,0.02);
        margin-bottom: 16px;
        position: relative;
        border: 1px solid #F1F5F9;
    }
    .health-score-num {
        font-size: 48px;
        font-weight: 800;
        background: linear-gradient(135deg, #10B981 0%, #059669 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        display: inline-block;
    }
    .health-score-pct {
        font-size: 20px;
        color: #10B981;
        font-weight: 800;
    }
    .progress-bar-container {
        background: #E2E8F0;
        border-radius: 999px;
        height: 10px;
        margin-top: 12px;
        width: 100%;
        overflow: hidden;
    }
    .progress-bar-fill {
        background: linear-gradient(90deg, #10B981, #059669);
        height: 100%;
        border-radius: 999px;
        width: 95%;
        transition: width 0.4s ease-in-out;
    }
    .chip-container {
        display: flex;
        gap: 6px;
        margin-top: 18px;
        flex-wrap: nowrap !important;
        justify-content: space-between;
        width: 100%;
    }
    .status-chip {
        padding: 6px 8px;
        border-radius: 999px;
        font-size: 11.5px;
        font-weight: 600;
        display: flex;
        align-items: center;
        justify-content: center;
        gap: 4px;
        border: 1px solid transparent;
        flex: 1;
        white-space: nowrap;
    }
    .chip-ph { background: #EFF6FF; color: #2563EB; border-color: #DBEAFE; }
    .chip-humidity { background: #ECFEFF; color: #0891B2; border-color: #CFFAFE; }
    .chip-temp { background: #FFF7ED; color: #EA580C; border-color: #FFEDD5; }

    .mini-grid {
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 12px;
    }
    .mini-card {
        background: #FFFFFF;
        border-radius: 18px;
        padding: 16px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.015);
        border: 1px solid #F1F5F9;
    }
    .mini-card-title { font-size: 11px; color: #94A3B8; text-transform: uppercase; font-weight: 700; letter-spacing: 0.5px; }
    .mini-card-val { font-size: 18px; font-weight: 800; margin-top: 6px; }
    .val-good { color: #2563EB; }
    .val-optimal { color: #EA580C; }

    /* Diagnosis Screen Elements */
    .leaf-img-placeholder {
        width: 100%;
        height: 200px;
        background: url('https://images.unsplash.com/photo-1592417817098-8f3d6eb19675?q=80&w=600') center/cover;
        border-radius: 20px;
        position: relative;
        margin-bottom: 16px;
        overflow: hidden;
        box-shadow: 0 4px 15px rgba(0,0,0,0.05);
    }
    .bounding-box {
        position: absolute;
        top: 25%; left: 40%; width: 90px; height: 90px;
        border: 3px solid #EAB308;
        border-radius: 14px;
        background: rgba(234, 179, 8, 0.15);
        box-shadow: 0 0 15px rgba(234, 179, 8, 0.3);
    }
    .box-label {
        position: absolute;
        top: -26px; left: -3px;
        background: #EAB308;
        color: white; font-size: 10px; font-weight: 800;
        padding: 3px 8px; border-radius: 6px;
        white-space: nowrap;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }
    .treatment-box {
        background: #F0FDF4;
        border: 1px solid #BBF7D0;
        border-radius: 14px;
        padding: 14px;
        margin-top: 14px;
    }

    /* Progress & Gamification Elements */
    .level-fill { background: linear-gradient(90deg, #8B5CF6, #EC4899); width: 77.5%; }
    .star-rating { color: #FBBF24; font-size: 22px; margin-top: 10px; }
    .leaderboard-row {
        display: flex; justify-content: space-between; align-items: center;
        padding: 12px 16px; border-radius: 14px; margin-bottom: 8px;
        background: #F8FAFC;
        border: 1px solid #F1F5F9;
    }
    .leaderboard-row span {
        color: #334155 !important;
        font-weight: 600 !important;
    }
    .leaderboard-active {
        background: #FEF08A !important;
        border: 1px solid #FDE047 !important;
        box-shadow: 0 4px 12px rgba(253, 224, 71, 0.15) !important;
    }
    .leaderboard-active span {
        color: #713F12 !important;
        font-weight: 800 !important;
    }

    /* Native Widget Styling overrides */
    .lemon-app .widget-button,
    .lemon-app .jupyter-button {
        border-radius: 20px !important;
        font-family: 'Outfit', sans-serif !important;
        font-weight: 600 !important;
        transition: all 0.2s ease-in-out !important;
        height: 38px !important;
        border: none !important;
        box-shadow: 0 4px 12px rgba(0,0,0,0.03) !important;
        display: inline-flex !important;
        align-items: center !important;
        justify-content: center !important;
    }

    .lemon-app .widget-button:hover,
    .lemon-app .jupyter-button:hover {
        transform: translateY(-1px) !important;
        box-shadow: 0 6px 16px rgba(0,0,0,0.06) !important;
    }

    .lemon-app .widget-button:active,
    .lemon-app .jupyter-button:active {
        transform: translateY(0px) !important;
    }

    /* Primary and Accent buttons */
    .lemon-app .widget-button.mod-primary,
    .lemon-app .jupyter-button.mod-primary {
        background: linear-gradient(135deg, #3B82F6 0%, #1D4ED8 100%) !important;
        color: white !important;
        box-shadow: 0 4px 14px rgba(59, 130, 246, 0.25) !important;
    }
    .lemon-app .widget-button.mod-primary:hover,
    .lemon-app .jupyter-button.mod-primary:hover {
        box-shadow: 0 6px 20px rgba(59, 130, 246, 0.35) !important;
    }

    .lemon-app .widget-button.mod-success,
    .lemon-app .jupyter-button.mod-success {
        background: linear-gradient(135deg, #10B981 0%, #059669 100%) !important;
        color: white !important;
        box-shadow: 0 4px 14px rgba(16, 185, 129, 0.25) !important;
    }
    .lemon-app .widget-button.mod-success:hover,
    .lemon-app .jupyter-button.mod-success:hover {
        box-shadow: 0 6px 20px rgba(16, 185, 129, 0.35) !important;
    }

    .lemon-app .widget-button.mod-info,
    .lemon-app .jupyter-button.mod-info {
        background: linear-gradient(135deg, #0284C7 0%, #0369A1 100%) !important;
        color: white !important;
        box-shadow: 0 4px 14px rgba(2, 132, 199, 0.25) !important;
    }
    .lemon-app .widget-button.mod-info:hover,
    .lemon-app .jupyter-button.mod-info:hover {
        box-shadow: 0 6px 20px rgba(2, 132, 199, 0.35) !important;
    }

    /* Bottom Nav Bar buttons styling */
    .lemon-app .nav-btn,
    .lemon-app .nav-btn button,
    .lemon-app button.nav-btn {
        background: transparent !important;
        color: #64748B !important;
        border: none !important;
        border-radius: 20px !important;
        font-weight: 600 !important;
        font-size: 11px !important;
        padding: 4px 6px !important;
        transition: all 0.2s ease-in-out !important;
        box-shadow: none !important;
        height: 38px !important;
        flex: 1 1 0% !important;
        min-width: 0 !important;
        margin: 0 2px !important;
        display: inline-flex !important;
        align-items: center !important;
        justify-content: center !important;
        white-space: nowrap !important;
        overflow: hidden !important;
    }

    .lemon-app .nav-btn:hover,
    .lemon-app .nav-btn button:hover,
    .lemon-app button.nav-btn:hover {
        background: #F1F5F9 !important;
        color: #0F172A !important;
    }

    /* Active Nav Bar item */
    .lemon-app .nav-btn.mod-success,
    .lemon-app .nav-btn.mod-success button,
    .lemon-app button.nav-btn.mod-success {
        background: #E8F5E9 !important;
        color: #10B981 !important;
        font-weight: 700 !important;
        box-shadow: 0 4px 10px rgba(16, 185, 129, 0.12) !important;
    }

    /* Bottom bar container styling */
    .lemon-app .widget-box.widget-hbox {
        background-color: #FFFFFF !important;
        border-top: 1px solid #F1F5F9 !important;
        padding: 12px 6px !important;
    }

    /* File Upload Widget button */
    .lemon-app .widget-upload {
        width: 100% !important;
        margin-top: 5px !important;
    }
    .lemon-app .widget-upload label,
    .lemon-app .widget-upload .jupyter-button,
    .lemon-app .widget-upload .jupyter-button.mod-success {
        background: linear-gradient(135deg, #8B5CF6 0%, #6D28D9 100%) !important;
        color: white !important;
        border: none !important;
        border-radius: 20px !important;
        padding: 10px 20px !important;
        font-family: 'Outfit', sans-serif !important;
        font-weight: 600 !important;
        font-size: 14px !important;
        box-shadow: 0 4px 14px rgba(139, 92, 246, 0.25) !important;
        text-align: center !important;
        display: block !important;
        width: 100% !important;
        cursor: pointer !important;
        transition: all 0.2s ease-in-out !important;
        box-sizing: border-box !important;
    }
    .lemon-app .widget-upload label:hover,
    .lemon-app .widget-upload .jupyter-button:hover,
    .lemon-app .widget-upload .jupyter-button.mod-success:hover {
        box-shadow: 0 6px 20px rgba(139, 92, 246, 0.35) !important;
        transform: translateY(-1px) !important;
    }
    .lemon-app .widget-upload label:active,
    .lemon-app .widget-upload .jupyter-button:active,
    .lemon-app .widget-upload .jupyter-button.mod-success:active {
        transform: translateY(0px) !important;
    }

    /* Text Inputs styling */
    .lemon-app .widget-text input {
        background: #FFFFFF !important;
        border: 1px solid #E2E8F0 !important;
        border-radius: 12px !important;
        padding: 10px 14px !important;
        font-family: 'Outfit', sans-serif !important;
        font-size: 14px !important;
        color: #0F172A !important;
        transition: all 0.2s ease-in-out !important;
        box-shadow: inset 0 2px 4px rgba(0,0,0,0.02) !important;
    }
    .lemon-app .widget-text input:focus {
        border-color: #3B82F6 !important;
        box-shadow: 0 0 0 3px rgba(59, 130, 246, 0.15), inset 0 2px 4px rgba(0,0,0,0.02) !important;
        outline: none !important;
    }

    /* Search Results style improvements */
    .main-card a {
        color: #0F172A !important;
        transition: color 0.15s ease !important;
    }
    .main-card a:hover {
        color: #10B981 !important;
    }
</style>
"""
display(HTML(css_style))

# 2. STATE MANAGEMENT
app_state = {
    "current_tab": "Home",
    "xp_score": 1550,
    "task_completed": [False, False, False],
    "sensor_ph": 6.20,
    "sensor_humidity": 45,
    "sensor_temp": 24,
    "history_ph": [6.1, 6.15, 6.22, 6.18, 6.12, 6.19],
    "history_humidity": [42, 44, 46, 45, 43, 45],
    "history_temp": [23, 24, 25, 24, 23, 24],
    "uploaded_image_html": ""
}

output_view = widgets.Output()


# 3. DYNAMIC CHART GENERATORS
def generate_base64_plot(title, y_label, values, y_limits, color):
    plt.figure(figsize=(4.8, 2.4), facecolor='none')
    days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

    plt.plot(days, values, color=color, marker='o', linewidth=2.5, markersize=6)
    plt.ylim(y_limits[0], y_limits[1])
    plt.ylabel(y_label, fontsize=8, color='#64748B')
    plt.title(title, fontsize=10, fontweight='bold', color='#0F172A', pad=10)
    plt.grid(axis='y', linestyle='--', alpha=0.3)

    ax = plt.gca()
    ax.set_facecolor('none')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#CBD5E1')
    ax.spines['bottom'].set_color('#CBD5E1')
    ax.tick_params(colors='#64748B', labelsize=8)

    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', dpi=150, transparent=True)
    plt.close()
    return base64.b64encode(buf.getvalue()).decode('utf-8')

# 4. VIEW RENDERERS
def render_home_screen():
    return f"""
    <div class="app-container">
        <div class="app-header">
            <h1 class="app-title">Hello Farmer!</h1>
            <p class="app-subtitle">Welcome back to your grove</p>
        </div>
        <div class="main-card">
            <div style="display:flex; justify-content:space-between; align-items:center;">
                <div>
                    <span style="font-size:15px; font-weight:bold; color:#0F172A;">My Lemon Tree</span><br>
                    <span style="font-size:11px; color:#94A3B8;">Last checked: Today</span>
                </div>
                <div style="background:#E8F5E9; border-radius:50%; padding:6px; color:#00C853; font-weight:bold;">📈</div>
            </div>
            <div style="margin-top:14px;">
                <div class="health-score-num">95<span class="health-score-pct">%</span></div>
                <div style="font-size:12px; color:#64748B; margin-top:-4px;">Health Score</div>
            </div>
            <div class="progress-bar-container"><div class="progress-bar-fill"></div></div>
            <div class="chip-container">
                <div class="status-chip chip-ph">🧬 pH: {app_state["sensor_ph"]:.1f}</div>
                <div class="status-chip chip-humidity">💧 Humidity: {app_state["sensor_humidity"]}%</div>
                <div class="status-chip chip-temp">🌡️ Temp: {app_state["sensor_temp"]}°C</div>
            </div>
        </div>
        <div class="mini-grid">
            <div class="mini-card">
                <div class="mini-card-title">Water Level</div>
                <div class="mini-card-val val-good">Good</div>
            </div>
            <div class="mini-card">
                <div class="mini-card-title">Sunlight</div>
                <div class="mini-card-val val-optimal">Optimal</div>
            </div>
        </div>
    </div>
    """

def render_diagnosis_screen():
    # Check if we have an uploaded image, otherwise use the default placeholder
    img_display = app_state.get("uploaded_image_html", "")
    if not img_display:
        img_display = """
        <div class="leaf-img-placeholder">
            <div class="bounding-box"><div class="box-label">Issue Detected</div></div>
        </div>
        """

    return f"""
    <div class="app-container">
        <div class="app-header">
            <h1 class="app-title">AI Leaf Diagnosis</h1>
            <p class="app-subtitle">Scan results</p>
        </div>
        {img_display}
        <div class="main-card">
            <div style="display:flex; align-items:center; gap:10px;">
                <div style="background:#FEE2E2; border-radius:50%; width:32px; height:32px; display:flex; align-items:center; justify-content:center; color:#EF4444; font-weight:bold;">!</div>
                <div>
                    <span style="font-weight:bold; color:#0F172A; font-size:15px;">Diagnosis: Citrus Leafminer</span><br>
                    <span style="font-size:12px; color:#64748B;">Detected with 92% confidence</span>
                </div>
            </div>
            <p style="font-size:13px; color:#334155; margin-top:12px; line-height:1.5;">
                <b>About this condition:</b> Citrus leafminer is a common pest that creates winding tunnels in young leaves.
            </p>
            <div class="treatment-box">
                <span style="color:#16A34A; font-weight:bold; font-size:13px;">🌿 Treatment</span>
                <p style="font-size:12px; color:#166534; margin: 4px 0 0 0; line-height:1.4;">Prune affected leaves and apply organic oil. Spray early in the morning or late evening for best results.</p>
            </div>
        </div>
    </div>
    """

def render_search_screen():
    return """
    <div style="padding: 20px 20px 15px 20px; background: #F8FAFC;">
        <div class="app-header" style="margin: 0;">
            <h1 class="app-title" style="color: #0F172A;">Academic Knowledge</h1>
            <p class="app-subtitle" style="color: #64748B;">AI-powered research database</p>
        </div>
    </div>
    """

def render_goals_screen():
    checked_1 = "☑️" if app_state["task_completed"][0] else "🔲"
    checked_2 = "☑️" if app_state["task_completed"][1] else "🔲"
    checked_3 = "☑️" if app_state["task_completed"][2] else "🔲"

    return f"""
    <div class="app-container">
        <div class="app-header">
            <h1 class="app-title">Your Progress</h1>
            <p class="app-subtitle">Keep growing your skills</p>
        </div>
        <div class="main-card">
            <div style="display:flex; align-items:center; gap:12px;">
                <div style="font-size:24px;">🏆</div>
                <div style="flex-grow:1;">
                    <span style="font-weight:bold; color:#0F172A; font-size:14px;">Level 5: Master of the Grove</span><br>
                    <span style="font-size:11px; color:#64748B;">450 XP to next level</span>
                </div>
                <div style="font-size:12px; font-weight:bold; color:#64748B;">{app_state["xp_score"]} / 2000 XP</div>
            </div>
            <div class="progress-bar-container"><div class="progress-bar-fill level-fill"></div></div>
            <div class="star-rating">⭐⭐⭐⭐<span style="color:#CBD5E1;">⭐</span></div>
        </div>

        <div style="margin-bottom:16px;">
            <h3 style="font-size:14px; font-weight:bold; color:#0F172A; margin-bottom:8px;">Daily Tasks</h3>
            <div class="main-card" style="padding:12px 16px; margin-bottom:0;">
                <div style="display:flex; justify-content:space-between; margin-bottom:10px; font-size:13px; color:#334155;">
                    <span>{checked_1} <span style="{ 'text-decoration: line-through; color: #94A3B8;' if app_state['task_completed'][0] else '' }">Scan your tree</span></span>
                    <span style="color:#22C55E; font-weight:bold;">+50 XP</span>
                </div>
                <div style="display:flex; justify-content:space-between; margin-bottom:10px; font-size:13px; color:#334155;">
                    <span>{checked_2} <span style="{ 'text-decoration: line-through; color: #94A3B8;' if app_state['task_completed'][1] else '' }">Check soil pH</span></span>
                    <span style="color:#22C55E; font-weight:bold;">+50 XP</span>
                </div>
                <div style="display:flex; justify-content:space-between; font-size:13px; color:#334155;">
                    <span>{checked_3} <span style="{ 'text-decoration: line-through; color: #94A3B8;' if app_state['task_completed'][2] else '' }">Add fertilizer</span></span>
                    <span style="color:#22C55E; font-weight:bold;">+50 XP</span>
                </div>
            </div>
        </div>

        <div>
            <h3 style="font-size:14px; font-weight:bold; color:#0F172A; margin-bottom:8px;">Leaderboard</h3>
            <div class="main-card" style="padding:10px; margin-bottom:0;">
                <div class="leaderboard-row leaderboard-active">
                    <span style="font-size:13px; font-weight:bold;">🥇 1. Oneel (You)</span>
                    <span style="font-size:13px; font-weight:bold;">2,450</span>
                </div>
                <div class="leaderboard-row">
                    <span style="font-size:13px; color:#475569;">🥈 2. Danny</span>
                    <span style="font-size:13px; font-weight:600; color:#475569;">2,280</span>
                </div>
                <div class="leaderboard-row">
                    <span style="font-size:13px; color:#475569;">🥉 3. Sarah</span>
                    <span style="font-size:13px; font-weight:600; color:#475569;">2,100</span>
                </div>
            </div>
        </div>
    </div>
    """

def render_analytics_screen():
    # Construct complete 7-day chronological sequences combining historical live array slices and active sensor numbers
    current_ph_history = app_state["history_ph"] + [app_state["sensor_ph"]]
    current_hum_history = app_state["history_humidity"] + [app_state["sensor_humidity"]]
    current_temp_history = app_state["history_temp"] + [app_state["sensor_temp"]]

    # Render line chart base64 image instances dynamically using the verified live streams
    ph_chart = generate_base64_plot("pH Levels - Past Week", "pH Scale", current_ph_history, [4, 9], '#00C853')
    humidity_chart = generate_base64_plot("Humidity - Past Week", "Percentage (%)", current_hum_history, [0, 100], '#0284C7')
    temp_chart = generate_base64_plot("Temperature - Past Week", "Degrees (°C)", current_temp_history, [0, 50], '#EA580C')

    # =======================================================
    # BUSINESS LOGIC LAYER: REAL-TIME DATA CRITERIA EVALUATION
    # =======================================================

    # 1. Dynamic Soil pH Diagnostic Matrix
    ph_val = app_state["sensor_ph"]
    if 6.0 <= ph_val <= 7.0:
        ph_status = f"Optimal at {ph_val:.2f}"
        ph_style = "color: #10B981;"
        ph_subtext = "Soil acidity matches parameters for efficient nutrient absorption."
    else:
        ph_status = f"Alert: Out of Range ({ph_val:.2f})"
        ph_style = "color: #EF4444; font-weight: 800;"
        ph_subtext = "Action Required: Alkaline/Acidic anomaly detected. Adjust soil buffering."

    # 2. Dynamic Atmospheric Humidity Diagnostic Matrix
    hum_val = app_state["sensor_humidity"]
    if 40 <= hum_val <= 60:
        hum_status = f"Good at {hum_val}%"
        hum_style = "color: #2563EB;"
        hum_subtext = "Ambient evaporation values promote healthy transpiration loops."
    else:
        hum_status = f"Unstable at {hum_val}%"
        hum_style = "color: #EA580C; font-weight: 800;"
        hum_subtext = "Humidity stress detected. Inspect automated micro-irrigation lines."

    # 3. Dynamic Telemetry Temperature Diagnostic Matrix
    temp_val = app_state["sensor_temp"]
    if 20 <= temp_val <= 30:
        temp_status = f"Ideal at {temp_val}°C"
        temp_style = "color: #EA580C;"
        temp_subtext = "Perfect microclimate range for optimal grove cell growth metabolic speeds."
    else:
        temp_status = f"Warning: High Stress ({temp_val}°C)"
        temp_style = "color: #EF4444; font-weight: 800;"
        temp_subtext = "Thermal threshold breach detected. Deploy shading structures if condition persists."

    # Return premium styled HTML framework with dynamically generated data cards
    return f"""
    <div class="app-container">
        <div class="app-header">
            <h1 class="app-title">Data Analytics</h1>
            <p class="app-subtitle">Track your tree's health metrics</p>
        </div>

        <div class="main-card" style="padding: 12px; margin-bottom: 12px;">
            <div class="mini-card-title">Soil pH Analysis</div>
            <div style="font-size: 18px; font-weight: bold; {ph_style} margin-top:2px;">{ph_status}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px; font-weight:600;">{ph_subtext} (Target: 6.0-7.0)</div>
        </div>

        <div class="main-card" style="padding: 12px; margin-bottom: 12px;">
            <div class="mini-card-title">Humidity Tracking</div>
            <div style="font-size: 18px; font-weight: bold; {hum_style} margin-top:2px;">{hum_status}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px; font-weight:600;">{hum_subtext} (Target: 40-60%)</div>
        </div>

        <div class="main-card" style="padding: 12px; margin-bottom: 16px;">
            <div class="mini-card-title">Ambient Temperature</div>
            <div style="font-size: 18px; font-weight: bold; {temp_style} margin-top:2px;">{temp_status}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px; font-weight:600;">{temp_subtext} (Target: 20-30°C)</div>
        </div>

        <div class="main-card" style="text-align:center; padding: 10px; margin-bottom:12px;">
            <img src="data:image/png;base64,{ph_chart}" style="max-width:100%; border-radius:10px;" />
        </div>
        <div class="main-card" style="text-align:center; padding: 10px; margin-bottom:12px;">
            <img src="data:image/png;base64,{humidity_chart}" style="max-width:100%; border-radius:10px;" />
        </div>
        <div class="main-card" style="text-align:center; padding: 10px;">
            <img src="data:image/png;base64,{temp_chart}" style="max-width:100%; border-radius:10px;" />
        </div>
    </div>
    """



# 4.5 SEARCH ENGINE BACKEND (FIREBASE & NLP)
FIREBASE_INDEX_URL = "https://lemonpulse-index-default-rtdb.europe-west1.firebasedatabase.app/lemon_disease_index.json"

# Documents mapping is defined below using integer keys

def process_query(query):
    """Tokenize query, remove stop words, and apply Porter Stemmer."""
    words = re.findall(r'\b[a-zA-Z]{3,}\b', query.lower())
    stop_words = {"the", "is", "at", "which", "on", "and", "of", "to", "in", "a", "for", "with", "as", "by", "an", "that", "from", "this", "it", "are", "was", "were"}
    stemmer = PorterStemmer()

    cleaned = []
    for word in words:
        if word not in stop_words:
            stemmed = stemmer.stem(word)
            if len(stemmed) >= 3:
                cleaned.append(stemmed)
    return cleaned

def search_documents(query):
    """Fetch Firebase index, match query terms, and rank results."""
    # 1. Fetch live index from your Firebase
    try:
        response = requests.get(FIREBASE_INDEX_URL, timeout=10)
        if response.status_code != 200:
            return []
        index_data = response.json()
    except Exception as e:
        print(f"Error fetching index: {e}")
        return []

    if not index_data or not isinstance(index_data, dict):
        return []

    # 2. Process query
    query_terms = process_query(query)
    if not query_terms:
        return []

    # 3. Rank documents (Matches * 100 + Frequency)
    doc_scores = defaultdict(lambda: {'matches': 0, 'frequency': 0, 'term_frequencies': {}})

    for term in query_terms:
        if term in index_data:
            term_data = index_data[term]

            if isinstance(term_data, dict) and "DocsIds" in term_data:
                docs_data = term_data["DocsIds"]

                # --- FIX: Handle Firebase returning a DICTIONARY ---
                if isinstance(docs_data, dict):
                    for doc_id, freq in docs_data.items():
                        if freq is not None:
                            doc_id_str = str(doc_id)
                            doc_scores[doc_id_str]['matches'] += 1
                            doc_scores[doc_id_str]['frequency'] += int(freq)
                            doc_scores[doc_id_str]['term_frequencies'][term] = int(freq)

                # --- FIX: Handle Firebase returning a LIST ---
                elif isinstance(docs_data, list):
                    for doc_id, freq in enumerate(docs_data):
                        # Firebase inserts 'null' (None in Python) for missing indexes
                        if freq is not None:
                            doc_id_str = str(doc_id)
                            doc_scores[doc_id_str]['matches'] += 1
                            doc_scores[doc_id_str]['frequency'] += int(freq)
                            doc_scores[doc_id_str]['term_frequencies'][term] = int(freq)

    # 4. Format results
    results = []
    for doc_id, scores in doc_scores.items():
        doc_key = int(doc_id) if doc_id.isdigit() else doc_id
        if doc_key in DOCUMENTS:
            relevance = (scores['matches'] * 100) + scores['frequency']

            # Slide 18 Rank: 1 - Product(1/Freq)
            if scores['matches'] > 0:
                prod = 1.0
                for term, freq in scores['term_frequencies'].items():
                    if freq > 0:
                        prod *= (1.0 / freq)
                tutorial_rank = 1.0 - prod
            else:
                tutorial_rank = 0.0

            results.append({
                'id': doc_key,
                'title': DOCUMENTS[doc_key]['title'],
                'url': DOCUMENTS[doc_key]['url'],
                'domain': DOCUMENTS[doc_key].get('domain', 'Citrus Pathology & Agronomy'),
                'summary': DOCUMENTS[doc_key].get('summary', ''),
                'score': relevance,
                'matches': scores['matches'],
                'frequency': scores['frequency'],
                'tutorial_rank': tutorial_rank,
                'term_frequencies': scores['term_frequencies']
            })

    # Sort by relevance (highest first)
    results.sort(key=lambda x: x['score'], reverse=True)
    return results


## Updated verified citrus and lemon articles mapping to prevent missing keys
DOCUMENTS = {
    1: {
        "title": "Citrus diseases detection using innovative deep learning approach and Hybrid Meta-Heuristic",
        "url": "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0316081",
        "domain": "Plant Pathology & AI",
        "summary": "Explores accurate deep learning computer vision frameworks to recognize early-stage pathogenic symptoms on citrus leaves."
    },
    2: {
        "title": "Advanced methods of plant disease detection",
        "url": "https://link.springer.com/article/10.1007/s13593-014-0246-1",
        "domain": "Agricultural Sustainability",
        "summary": "A review of hyperspectral sensing, optical sensors, and automated indexing engines used in crop protection."
    },
    3: {
        "title": "Influence of soil temperature, moisture, and pH on citrus growth",
        "url": "https://link.springer.com/article/10.1007/BF02215619",
        "domain": "Agronomy & Soil Science",
        "summary": "Analyses how steady soil acidities impact microclimate nutrient delivery in commercial citrus crops."
    },
    4: {
        "title": "Automatic Detection of Citrus Fruit and Leaves Diseases Using Deep Neural Network Model",
        "url": "https://ieeexplore.ieee.org/document/9481921",
        "domain": "Computer Vision Applications",
        "summary": "Implements fine-tuned convolutional architectures focused on isolating and detecting citrus leafminer trails dynamically."
    },
    5: {
        "title": "Image Recognition of Citrus Diseases Based on Deep Learning",
        "url": "https://www.techscience.com/cmc/v66n1/40458",
        "domain": "Precision Agriculture",
        "summary": "Focuses on building lightweight neural network layers for edge hardware deployment to assist farmers in real-time grove diagnostics."
    }
}

# Universal Layout Elements for the Advanced Search Viewport
search_input = widgets.Text(
    value="citrus disease",
    placeholder="Type keywords (e.g., pH, leafminer, irrigation)...",
    layout=widgets.Layout(width='100%', margin='0 0 10px 0')
)
search_btn = widgets.Button(
    description="🔍 Search Database & Generate Advice",
    button_style="primary",
    layout=widgets.Layout(width='100%', height='40px')
)
rag_result_out = widgets.Output()

search_box_ui = widgets.VBox(
    [search_input, search_btn, rag_result_out],
    layout=widgets.Layout(padding='0 20px 20px 20px')
)
search_box_ui.add_class('lemon-app')

def execute_rag_search(b):
    """Coordinates search pipeline, fallback smoothly to title-matching via Cerebras REST API if index is empty."""
    with rag_result_out:
        clear_output()
        query = search_input.value.strip()

        if not query:
            display(HTML("<div style='color:#EF4444; padding:10px 0; font-size:13px;'>⚠️ Please enter a search query.</div>"))
            return

        display(HTML(f"<div style='color:#2563EB; padding:10px 0; font-size:13px; font-weight:bold;'>🔄 Checking Firebase Index rules for '{query}'...</div>"))

        # Call your core search algorithm that queries the Firebase Inverted Index
        results = search_documents(query)
        clear_output()

        is_fallback_activated = False
        source_label_text = "Primary Index Ground-Truth Match"

        if results:
            # Case A: Exact keyword mapping found inside Firebase index node
            best_document = results[0]
            doc_title = best_document.get('title', 'Untitled Citrus Publication')
            doc_domain = best_document.get('domain', 'Citrus Pathology & Agronomy')
            doc_summary = best_document.get('summary', '')
            doc_url = best_document.get('url', '#')
            doc_score = best_document.get('score', 0)
            doc_matches = best_document.get('matches', 1)
        else:
            # Case B: Direct keyword match missing, trigger AI semantic selection by titles logic
            is_fallback_activated = True
            source_label_text = "AI Semantic Fallback Engine (Logical Title Match)"
            display(HTML("<div style='color:#F59E0B; padding:5px 0; font-size:13px; font-weight:bold;'>ℹ️ Term not found in Inverted Index. Activating AI Semantic Title-Matching Fallback...</div>"))

        display(HTML("<div style='color:#16A34A; padding:10px 0; font-size:13px; font-weight:bold;'>🔄 Connecting directly to Cerebras Cloud Engine... Generating smart response...</div>"))

        if not is_fallback_activated:
            rag_prompt = f"""
            You are an expert AI Agronomist for the 'Lemon Pulse' smart orchard system.
            Provide a concise, practical answer to the user's question: "{query}" based ONLY on the following verified scientific publication context:

            Document Title: {doc_title}
            Scientific Domain: {doc_domain}
            Academic Summary: {doc_summary}

            Format your response for a farmer: explain the context clearly in 2-3 sentences, and add bullet points for actionable steps if applicable. Keep it professional. Do not mention any JSON tags.
            """
        else:
            compiled_articles_context = ""
            for doc_id, meta in DOCUMENTS.items():
                compiled_articles_context += f"\nDocument ID {doc_id}:\nTitle: {meta['title']}\nDomain: {meta['domain']}\nSummary: {meta['summary']}\n---"

            rag_prompt = f"""
            You are an expert AI Agronomist for the 'Lemon Pulse' smart orchard system.
            The user asked: "{query}".
            No direct keyword matches were found in our inverted index. Review the following 5 citrus-related articles and logically select the SINGLE most relevant article based on its title and context to answer the question.

            Available Core Articles:{compiled_articles_context}

            Based on your logical selection, perform two tasks:
            1. Generate a concise, practical answer (2-3 sentences + bullet points) using the summary text of your selected article.
            2. Append a clean structured configuration tag at the absolute end of your response, exactly like this: [SELECTED_DOC_ID: X] where X is the picked document number (1, 2, 3, 4, or 5). Do not write anything else inside that tag.
            """

        # --- DIRECT HTTP POST TO CEREBRAS ENDPOINT (OpenAI-Compatible Routing) ---
        cerebras_key = "csk-jc54vyvvjctwfjhvv854f6w6wrwp9tnhvj3hywe44dvxfftt"
        api_url = "https://api.cerebras.ai/v1/chat/completions"

        headers = {
            "Authorization": f"Bearer {cerebras_key}",
            "Content-Type": "application/json"
        }

        # Array of currently supported active production models on Cerebras endpoints (Fixes the 404 error)
        candidate_models = ["llama-3.3-70b", "zai-glm-4.7"]
        generated_response_text = ""

        # Fallback loop to try alternative model IDs sequentially if one is missing or under maintenance
        for model_id in candidate_models:
            payload = {
                "model": model_id,
                "messages": [
                    {"role": "system", "content": "You are a precise agricultural consultant. Synthesize verified data accurately based on titles and contexts."},
                    {"role": "user", "content": rag_prompt}
                ],
                "temperature": 0.2
            }
            try:
                response = requests.post(api_url, headers=headers, json=payload, timeout=15)
                if response.status_code == 200:
                    response_json = response.json()
                    generated_response_text = response_json['choices'][0]['message']['content']
                    break # Break out of loop immediately once response packages register successfully
            except Exception:
                pass

        # If a response text was captured successfully, clear logs and render custom HTML grids templates
        if generated_response_text:
            clear_output()

            if is_fallback_activated:
                selected_id = 1
                tag_match = re.search(r'\[SELECTED_DOC_ID:\s*(\d+)\]', generated_response_text)
                if tag_match:
                    selected_id = int(tag_match.group(1))
                    generated_response_text = re.sub(r'\[SELECTED_DOC_ID:\s*\d+\]', '', generated_response_text).strip()

                fallback_meta = DOCUMENTS.get(selected_id, DOCUMENTS[1])
                doc_title = fallback_meta['title']
                doc_domain = fallback_meta['domain']
                doc_url = fallback_meta['url']
                doc_score = "N/A (Semantic Choice)"
                doc_matches = 0
            else:
                selected_id = best_document['id']

            badge_color = "#10B981" if not is_fallback_activated else "#F59E0B"
            badge_bg = "#F0FDF4" if not is_fallback_activated else "#FFFBEB"
            badge_border = "#DCFCE7" if not is_fallback_activated else "#FEF3C7"
            badge_text_color = "#15803D" if not is_fallback_activated else "#B45309"

            rag_block_html = f"""
            <div class="main-card" style="background: {badge_bg}; border: 1px solid {badge_border}; border-left: 5px solid {badge_color}; margin-bottom: 24px; text-align: left; padding: 20px; box-shadow: 0 4px 12px rgba(0,0,0,0.02);">
                <div style="font-size: 11px; font-weight: 800; color: {badge_text_color}; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 6px;">🧠 {source_label_text}</div>
                <p style="font-size: 13.5px; color: #14532D; margin: 0; line-height: 1.5; font-weight: 500;">{generated_response_text}</p>
            </div>
            <div style='font-size:13px; font-weight:bold; color:#64748B; margin-bottom: 12px; text-align: left; padding-left: 4px;'>
                📄 Primary Ground-Truth Publication Reference Utilized:
            </div>
            """
            display(HTML(rag_block_html))

            confidence_pct = 75 if is_fallback_activated else min(99, 50 + (best_document['score'] * 2))
            card_accent_color = "#3B82F6" if not is_fallback_activated else "#F59E0B"

            source_card_html = f"""
            <div class="main-card" style="margin-bottom: 12px; border-left: 4px solid {card_accent_color}; text-align: left; padding: 18px; box-shadow: 0 4px 10px rgba(0,0,0,0.02);">
                <span style="font-size:14px; font-weight:bold; color:#0F172A;">
                    <a href="{doc_url}" target="_blank" style="color:#3B82F6; text-decoration:none;">📄 {doc_title}</a>
                </span><br>
                <div style="margin-top: 6px; font-size: 12.5px; color: #475569; font-weight: 500;">Category Domain: <i>{doc_domain}</i></div>
                <div style="background:#F8FAFC; padding:10px; border-radius:8px; margin-top:10px; font-size:12px; color:#64748B; border: 1px solid #F1F5F9;">
                    <b>Matches in Index:</b> {doc_matches} | <b>Calculated Index Priority Score:</b> {doc_score} pts
                </div>
                <div class="progress-bar-container" style="height:5px; margin-top:10px; background:#E2E8F0;">
                    <div class="progress-bar-fill" style="width:{confidence_pct}%; background: {card_accent_color};"></div>
                </div>
            </div>
            """
            display(HTML(source_card_html))

            # --- RENDER THE COMPLETE RANKED MATCHING PUBLICATIONS LIST ---
            all_docs_display = []
            for doc_id, meta in DOCUMENTS.items():
                all_docs_display.append({
                    'id': doc_id,
                    'title': meta['title'],
                    'url': meta['url'],
                    'domain': meta['domain'],
                    'score': 0,
                    'matches': 0,
                    'tutorial_rank': 0.0,
                    'term_frequencies': {}
                })

            if results:
                for res in results:
                    for doc in all_docs_display:
                        if doc['id'] == res['id']:
                            doc.update(res)

            # Sort: matched docs (by score) first, then non-matched
            all_docs_display.sort(key=lambda x: x['score'], reverse=True)

            publications_list_html = f"""
            <div style='font-size:13px; font-weight:bold; color:#64748B; margin-bottom: 12px; text-align: left; padding-left: 4px; margin-top: 24px;'>
                📊 All Academic Publications Ranked by Index Score:
            </div>
            """

            for doc in all_docs_display:
                is_selected_primary = (doc['id'] == selected_id)

                if is_selected_primary:
                    border_style = f"border-left: 5px solid {card_accent_color}; background: #F8FAFC;"
                    title_prefix = "⭐ [PRIMARY REF] "
                elif doc['score'] > 0:
                    border_style = "border-left: 4px solid #10B981; background: #FFFFFF;"
                    title_prefix = "✅ [MATCHED] "
                else:
                    border_style = "border-left: 4px solid #CBD5E1; background: #FFFFFF; opacity: 0.75;"
                    title_prefix = "📄 [NO MATCH] "

                if doc['score'] > 0:
                    matched_terms_str = ", ".join([f"'{t}': {f}" for t, f in doc['term_frequencies'].items()])
                    term_details = f"<div style='margin-top: 6px; font-size: 11.5px; color: #15803D;'><b>Matching Terms Freqs:</b> {matched_terms_str}</div>"
                else:
                    term_details = "<div style='margin-top: 6px; font-size: 11.5px; color: #64748B;'>No query terms matched in the inverted index.</div>"

                doc_matches = doc['matches']
                doc_score_text = f"{doc['score']} pts" if doc['score'] > 0 else "0 pts"
                tutorial_rank_text = f"{doc['tutorial_rank'] * 100:.1f}%" if doc['score'] > 0 else "0.0%"

                publications_list_html += f"""
                <div class="main-card" style="margin-bottom: 12px; padding: 14px; box-shadow: 0 2px 6px rgba(0,0,0,0.01); {border_style}">
                    <span style="font-size:13px; font-weight:bold; color:#0F172A;">
                        <a href="{doc['url']}" target="_blank" style="color:#1E40AF; text-decoration:none;">{title_prefix}{doc['title']}</a>
                    </span><br>
                    <div style="margin-top: 4px; font-size: 11.5px; color: #64748B;">Category Domain: <i>{doc['domain']}</i></div>
                    {term_details}
                    <div style="background:#F8FAFC; padding:8px; border-radius:6px; margin-top:8px; font-size:11px; color:#475569; border: 1px solid #F1F5F9; display: flex; justify-content: space-between;">
                        <span><b>Matches:</b> {doc_matches}</span>
                        <span><b>Priority Score:</b> {doc_score_text}</span>
                        <span><b>Rank:</b> {tutorial_rank_text}</span>
                    </div>
                </div>
                """

            display(HTML(publications_list_html))
        else:
            clear_output()
            display(HTML(f"""
            <div class="main-card" style="margin-top: 15px; border-left: 4px solid #EF4444; background: #FEF2F2;">
                <span style="font-size:14px; font-weight:bold; color:#991B1B;">Cerebras Cloud Handshake Failure</span><br>
                <span style="font-size:12px; color:#7F1D1D;">All model options returned errors. Please verify credentials status or net infrastructure loops.</span>
            </div>
            """))

search_btn.on_click(execute_rag_search)

# 6. INTERACTIVE HARDWARE & ACTION ENGINES
action_btn_home = widgets.Button(description="🔄 Sample IoT Sensors Now (ESP32 Polling)", button_style="info", layout=widgets.Layout(width='100%', margin='10px 0 0 0'))

file_upload_widget = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='📷 Upload Leaf Photo',
    button_style='success',
    layout=widgets.Layout(width='100%', margin='5px 0 0 0')
)

action_btn_diag = widgets.Button(description="📥 Add to Tasks", button_style="success", icon="check", layout=widgets.Layout(width='100%', margin='10px 0 0 0'))

def on_sample_clicked(b):
    import requests
    import random

    # Central proxy server URL node provided by the instructor
    SERVER_BASE_URL = "https://server-cloud-v645.onrender.com/history"

    # We request 7 samples: 6 to construct historical data vectors, 1 for the current active view
    sample_limit = 7

    print("🔄 Initializing secure connection sequence with cloud gateway bridge server...")
    print("⏳ Note: If the container is currently idle/sleeping, the initial handshake can take up to 30 seconds to wake up...")

    try:
        # 1. Pull Temperature Telemetry Array Packets
        temp_resp = requests.get(SERVER_BASE_URL, params={"feed": "temperature", "limit": sample_limit}, timeout=35)
        temp_json = temp_resp.json()

        # 2. Pull Humidity Telemetry Array Packets
        hum_resp = requests.get(SERVER_BASE_URL, params={"feed": "humidity", "limit": sample_limit}, timeout=35)
        hum_json = hum_resp.json()

        # 3. Pull Soil Analytics Data Array (Mapped to derive proportional pH variance scales)
        soil_resp = requests.get(SERVER_BASE_URL, params={"feed": "soil", "limit": sample_limit}, timeout=35)
        soil_json = soil_resp.json()

        # Validate that database response frames are returned properly from all three feeds
        if "data" in temp_json and "data" in hum_json and "data" in soil_json:
            raw_temp = [float(item["value"]) for item in temp_json["data"]]
            raw_hum  = [float(item["value"]) for item in hum_json["data"]]
            raw_soil = [float(item["value"]) for item in soil_json["data"]]

            # Safeguard structure: pad arrays with default baselines if the server returns fewer than 7 rows
            while len(raw_temp) < sample_limit: raw_temp.append(24.0)
            while len(raw_hum) < sample_limit:  raw_hum.append(45.0)
            while len(raw_soil) < sample_limit: raw_soil.append(65.0)

            # Reverse lists to enforce true chronological order (Oldest -> Newest) for Matplotlib canvas rendering
            raw_temp.reverse()
            raw_hum.reverse()
            raw_soil.reverse()

            # --- TELEMETRY CONVERSION: Normalize raw soil counts to a healthy citrus pH range (6.0 - 7.0) ---
            processed_ph = [6.0 + (val % 1.0 if val > 14 else val / 100.0) for val in raw_soil]
            processed_ph = [max(5.5, min(7.5, val)) for val in processed_ph]

            # Bind the latest chronological coordinate indices straight to global Application State metrics
            app_state["sensor_temp"]     = int(raw_temp[-1])
            app_state["sensor_humidity"] = int(raw_hum[-1])
            app_state["sensor_ph"]       = float(processed_ph[-1])

            # Commit preceding data arrays to update the Data Analytics historical trend lines
            app_state["history_temp"]     = [int(x) for x in raw_temp[:-1]]
            app_state["history_humidity"] = [int(x) for x in raw_hum[:-1]]
            app_state["history_ph"]       = [float(x) for x in processed_ph[:-1]]

            # --- CHOSEN FEATURE INTEGRATION: Smart Irrigation Weather Sync ---
            # Simulate real-time validation check against localized weather forecast parameters
            rain_forecast_12h = random.choice([0.0, 1.2, 5.8, 8.4])  # Expected rainfall metric in millimeters

            print("✅ Live cloud metrics synchronized successfully! Updating dashboard components...")
            if rain_forecast_12h >= 5.0:
                print(f"🌧️ [WEATHER SYNC ALERT] Significant incoming precipitation detected ({rain_forecast_12h}mm) within 12 hours!")
                print("🤖 Business Logic Layer has auto-postponed soil irrigation tasks to protect lemon tree roots from root rot.")
                app_state["task_completed"][1] = True  # Auto-complete/defer task requirements safely
            else:
                print("☀️ Local weather forecast indicates dry conditions. Standard check soil task entry verified and completed.")
                app_state["task_completed"][1] = True

            # Advance gamified progress metrics and append XP scoring structures
            app_state["xp_score"] = min(2000, app_state["xp_score"] + 50)

        else:
            print("⚠️ Server content response validation mismatch. Triggering static fallback structures.")

    except Exception as e:
        print(f"❌ Failed to communicate with Render server instance gateway: {e}")
        print("🔄 Reverting application state routine back to local simulation metrics.")

    # Re-fire Single Page Application rendering loops to display the updated interface immediately
    update_display()

def on_file_upload(change):
    uploaded_file = file_upload_widget.value
    if uploaded_file:
        file_info = uploaded_file[0] if isinstance(uploaded_file, tuple) else list(uploaded_file.values())[0]
        content = file_info['content']
        b64_img = base64.b64encode(content).decode('utf-8')

        # Save to state and route to diagnosis
        app_state["uploaded_image_html"] = f'<img src="data:image/jpeg;base64,{b64_img}" style="width:100%; height:200px; object-fit:cover; border-radius:16px; margin-bottom:16px;" />'
        trigger_tab_routing("Diagnosis")

# Connect the upload widget to the function
file_upload_widget.observe(on_file_upload, names='value')

def on_add_task_clicked(b):
    print("Writing state logs to cloud storage structure...")
    app_state["task_completed"][0] = True
    app_state["xp_score"] = min(2000, app_state["xp_score"] + 50)
    update_display()

action_btn_home.on_click(on_sample_clicked)
action_btn_diag.on_click(on_add_task_clicked)

# 7. ROUTER AND GLOBAL NAVIGATION ENGINE
def update_display():
    with output_view:
        clear_output(wait=True)
        if app_state["current_tab"] == "Home":
            display(HTML(render_home_screen()))
            # Action_btn_camera is now file_upload_widget
            display(widgets.VBox([action_btn_home]))
        elif app_state["current_tab"] == "Diagnosis":
            display(HTML(render_diagnosis_screen()))
            display(widgets.VBox([file_upload_widget, action_btn_diag]))
        elif app_state["current_tab"] == "Search":
            display(HTML(render_search_screen()))
            display(search_box_ui)
            execute_rag_search(None)
        elif app_state["current_tab"] == "Goals":
            display(HTML(render_goals_screen()))
        elif app_state["current_tab"] == "Analytics":
            display(HTML(render_analytics_screen()))

def trigger_tab_routing(target_tab):
    app_state["current_tab"] = target_tab
    for btn in [nav_home, nav_diag, nav_srch, nav_goal, nav_anal]:
        btn.button_style = ''
    if target_tab == "Home": nav_home.button_style = 'success'
    elif target_tab == "Diagnosis": nav_diag.button_style = 'success'
    elif target_tab == "Search": nav_srch.button_style = 'success'
    elif target_tab == "Goals": nav_goal.button_style = 'success'
    elif target_tab == "Analytics": nav_anal.button_style = 'success'
    update_display()

# Navigation Tabs Bar Elements
nav_home = widgets.Button(description="🏠 Home", layout=widgets.Layout(width='auto', flex='1'))
nav_home.add_class('nav-btn')
nav_diag = widgets.Button(description="🩺 Diagnosis", layout=widgets.Layout(width='auto', flex='1'))
nav_diag.add_class('nav-btn')
nav_srch = widgets.Button(description="🔍 Search", layout=widgets.Layout(width='auto', flex='1'))
nav_srch.add_class('nav-btn')
nav_goal = widgets.Button(description="🏆 Goals", layout=widgets.Layout(width='auto', flex='1'))
nav_goal.add_class('nav-btn')
nav_anal = widgets.Button(description="📊 Analytics", layout=widgets.Layout(width='auto', flex='1'))
nav_anal.add_class('nav-btn')

nav_home.on_click(lambda b: trigger_tab_routing("Home"))
nav_diag.on_click(lambda b: trigger_tab_routing("Diagnosis"))
nav_srch.on_click(lambda b: trigger_tab_routing("Search"))
nav_goal.on_click(lambda b: trigger_tab_routing("Goals"))
nav_anal.on_click(lambda b: trigger_tab_routing("Analytics"))

nav_home.button_style = 'success'
nav_bar = widgets.HBox([nav_home, nav_diag, nav_srch, nav_goal, nav_anal],
                       layout=widgets.Layout(justify_content='space-around', background_color='white', padding='10px 0', border_bottom='1px solid #E2E8F0'))
main_frame = widgets.VBox([nav_bar, output_view],
                          layout=widgets.Layout(max_width='900px',
                                                margin='0 auto',
                                                border='1px solid #CBD5E1',
                                                border_radius='12px',
                                                overflow='hidden',
                                                background_color='#F8FAFC'))
main_frame.add_class('lemon-app')


display(main_frame)
update_display()

🔄 Initializing secure connection sequence with cloud gateway bridge server...
⏳ Note: If the container is currently idle/sleeping, the initial handshake can take up to 30 seconds to wake up...
✅ Live cloud metrics synchronized successfully! Updating dashboard components...
☀️ Local weather forecast indicates dry conditions. Standard check soil task entry verified and completed.
🔄 Initializing secure connection sequence with cloud gateway bridge server...
⏳ Note: If the container is currently idle/sleeping, the initial handshake can take up to 30 seconds to wake up...
✅ Live cloud metrics synchronized successfully! Updating dashboard components...
☀️ Local weather forecast indicates dry conditions. Standard check soil task entry verified and completed.
🔄 Initializing secure connection sequence with cloud gateway bridge server...
⏳ Note: If the container is currently idle/sleeping, the initial handshake can take up to 30 seconds to wake up...
✅ Live cloud metrics synchronized successfull